<a href="https://colab.research.google.com/github/Anthea05/Anthea_machine-learning/blob/main/JS09/Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 8.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 22.1 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520038 sha256=1f2f434585dfffc71baf6ab88b1ba8ba4762f4c8e38e1167de479d603d944101
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679800 sha256=4f749db2198bbd6bafa151da519424384867d6c608c9837a67a18f77a12a8435
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [3]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# Kita bikin list kombinasi jumlah data dan dimensi sesuai soal
skenario = [
    (1000, 2),
    (1000, 5),
    (1000000, 2),
    (1000000, 5)
]

k = 5 # Kita cari 5 tetangga terdekat aja

for n_data, dim in skenario:
    print(f"\n=========================================")
    print(f"Mulai tes untuk {n_data} data dengan {dim} Dimensi")
    print(f"=========================================")

    # Bikin data dummy (tipe float32 biar support faiss)
    X = np.random.random((n_data, dim)).astype('float32')
    # Bikin 1 query point buat dites
    query = np.random.random((1, dim)).astype('float32')

    # ---------------------------------------------------
    # 1. ANNOY
    # ---------------------------------------------------
    annoy_idx = AnnoyIndex(dim, 'euclidean')
    for i in range(n_data):
        annoy_idx.add_item(i, X[i])

    # Mulai ngitung waktu build sama query
    start_build = time.time()
    annoy_idx.build(10) # pake 10 tree aja biar ga kelamaan
    waktu_build_annoy = time.time() - start_build

    start_query = time.time()
    annoy_idx.get_nns_by_vector(query[0], k)
    waktu_query_annoy = time.time() - start_query

    print(f"ANNOY -> Waktu Build: {waktu_build_annoy:.4f} detik | Query: {waktu_query_annoy:.6f} detik")

    # ---------------------------------------------------
    # 2. FAISS
    # ---------------------------------------------------
    faiss_idx = faiss.IndexFlatL2(dim)

    start_build = time.time()
    faiss_idx.add(X)
    waktu_build_faiss = time.time() - start_build

    start_query = time.time()
    faiss_idx.search(query, k)
    waktu_query_faiss = time.time() - start_query

    print(f"FAISS -> Waktu Build: {waktu_build_faiss:.4f} detik | Query: {waktu_query_faiss:.6f} detik")

    # ---------------------------------------------------
    # 3. HNSW
    # ---------------------------------------------------
    hnsw_idx = hnswlib.Index(space='l2', dim=dim)

    start_build = time.time()
    # M=16 dan ef_construction=100 itu parameter standar yang sering dipakai
    hnsw_idx.init_index(max_elements=n_data, ef_construction=100, M=16)
    hnsw_idx.add_items(X)
    waktu_build_hnsw = time.time() - start_build

    hnsw_idx.set_ef(50)
    start_query = time.time()
    hnsw_idx.knn_query(query, k=k)
    waktu_query_hnsw = time.time() - start_query

    print(f"HNSW  -> Waktu Build: {waktu_build_hnsw:.4f} detik | Query: {waktu_query_hnsw:.6f} detik")

# Catatan: Pas ngerun yang 1 juta data, sabar dikit ya nunggunya karena agak berat.


Mulai tes untuk 1000 data dengan 2 Dimensi
ANNOY -> Waktu Build: 0.0202 detik | Query: 0.000050 detik
FAISS -> Waktu Build: 0.0001 detik | Query: 0.000064 detik
HNSW  -> Waktu Build: 0.0312 detik | Query: 0.000054 detik

Mulai tes untuk 1000 data dengan 5 Dimensi
ANNOY -> Waktu Build: 0.0147 detik | Query: 0.000048 detik
FAISS -> Waktu Build: 0.0000 detik | Query: 0.000045 detik
HNSW  -> Waktu Build: 0.0349 detik | Query: 0.000062 detik

Mulai tes untuk 1000000 data dengan 2 Dimensi
ANNOY -> Waktu Build: 30.4645 detik | Query: 0.000085 detik
FAISS -> Waktu Build: 0.0021 detik | Query: 0.009875 detik
HNSW  -> Waktu Build: 55.1954 detik | Query: 0.000063 detik

Mulai tes untuk 1000000 data dengan 5 Dimensi
ANNOY -> Waktu Build: 18.6968 detik | Query: 0.000130 detik
FAISS -> Waktu Build: 0.0177 detik | Query: 0.027130 detik
HNSW  -> Waktu Build: 87.9768 detik | Query: 0.000101 detik
